# YOLO V4 — Fresh 20-Epoch Notebook
**This notebook prints visible status messages in each section.**
**YOLO11s · 800px · 20 epochs maximum · patience 10 · four classes (food, plate, utensil, wrapper).**
Use the unchanged `YOLO_V4_865_4class_audit_pending.zip` archive from the conversation. Choose T4 GPU and Run all. Primary training uses 521 cafeteria images; optional 161 legacy images are disabled by default. Test evaluation is disabled by default. Annotations remain provisional, with 15 images quarantined and a historical test set.


## 1. Install dependencies

In [ ]:
print('STEP 1/8: Checking GPU and installing training dependencies...', flush=True)
import subprocess,sys,torch,zipfile,hashlib,json
from pathlib import Path
assert torch.cuda.is_available(),'Runtime > Change runtime type > T4 GPU'
subprocess.run([sys.executable,'-m','pip','install','ultralytics==8.4.175','PyYAML>=6.0','Pillow>=10.0'],check=True)
print('GPU:',torch.cuda.get_device_name(0))

## 2. Upload the complete V4 dataset ZIP (not V2 or V3)

In [ ]:
print('STEP 2/8: Uploading and extracting the complete V4 ZIP...', flush=True)
from google.colab import files
expected='YOLO_V4_865_4class_audit_pending.zip'
expected_sha='95a386dbd9cf707e642237b92326a27ffd4958b893d1e0a7089e463b823c91e7'
archive=Path('/content')/expected
def sha(p):
    d=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''):d.update(block)
    return d.hexdigest()
if not archive.exists():
    print('Upload:',expected)
    upload=files.upload()
    assert expected in upload,'Wrong ZIP: upload '+expected
    archive.write_bytes(upload[expected])
assert sha(archive)==expected_sha,'ZIP checksum mismatch'
root=Path('/content/yolo_v4_four_class')
if not (root/'source_manifest.json').is_file():
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        assert all(Path(n.filename).parts[:1]==('yolo_v4_four_class',) for n in z.infolist())
        z.extractall('/content')
print('Extracted:',root)

## 3. Structural audit and annotation status

In [ ]:
print('STEP 3/8: Verifying 865 images and labels...', flush=True)
trainer=root/'train_yolo_v4.py'
subprocess.run([sys.executable,str(trainer),'verify','--root',str(root)],check=True)
print(json.loads((root/'audit_summary.json').read_text())['status'])
print('Review ledger:',root/'audit_queue.csv')
print('24 source photos have empty labels; 14 were quarantined from training. The nine val and one test negatives require an image-by-image decision.')

## 4. Train cafeteria-only YOLO11s at 800px — up to 20 epochs, patience 10

In [ ]:
print('STEP 4/8: Starting 20-epoch YOLO11s training; logs appear below.', flush=True)
caf=Path('/content/YOLO_V4_cafeteria_results')
assert not caf.exists(),'Refusing to overwrite a prior run'
subprocess.run([sys.executable,'-u',str(trainer),'train','--root',str(root),'--variant','cafeteria','--output',str(caf),'--device','0','--batch','4','--imgsz','800','--epochs','20','--patience','10'],check=True)

## 5. Optional comparison: add the 161 legacy images. Set RUN_SUPPLEMENTED=True to train.

In [ ]:
print('STEP 5/8: Optional comparison training...', flush=True)
RUN_SUPPLEMENTED=False
sup=Path('/content/YOLO_V4_supplemented_results')
if RUN_SUPPLEMENTED:
    assert not sup.exists(),'Refusing to overwrite prior results'
    subprocess.run([sys.executable,'-u',str(trainer),'train','--root',str(root),'--variant','supplemented','--output',str(sup),'--device','0','--batch','4','--imgsz','800','--epochs','20','--patience','10'],check=True)
else:
    print('Optional supplemented variant skipped')

## 6. Compare validation scores; choose based on validation, not test

In [ ]:
print('STEP 6/8: Reading validation metrics...', flush=True)
for name,folder in [('cafeteria',caf),('supplemented',sup)]:
    p=folder/'val_summary.json'
    if not p.is_file():continue
    m=json.loads(p.read_text())['metrics']
    print(name,'mAP50',round(m['mAP50'],4),'mAP50-95',round(m['mAP50_95'],4))
    for c in ['food','plate','utensil','wrapper']:
        print(' ',c,m['per_class'].get(c))

## 7. One-time test evaluation after selecting by validation and resolving audit flags. Disabled by default.

In [ ]:
print('STEP 7/8: Test evaluation (off by default)...', flush=True)
CHOSEN_VARIANT='cafeteria'
RUN_FINAL_TEST=False
selected={'cafeteria':caf,'supplemented':sup}[CHOSEN_VARIANT]
assert (selected/'val_summary.json').is_file()
if RUN_FINAL_TEST:
    subprocess.run([sys.executable,str(trainer),'test','--root',str(root),'--variant',CHOSEN_VARIANT,'--output',str(selected),'--device','0','--batch','4','--imgsz','800'],check=True)
else:
    print('TEST NOT EVALUATED: change RUN_FINAL_TEST to True only after model selection and test annotation review')

## 8. Download checkpoint and results

In [ ]:
print('STEP 8/8: Packaging results...', flush=True)
result_zip=Path('/content/YOLO_V4_model_results.zip')
with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for name,folder in [('cafeteria',caf),('supplemented',sup)]:
        for rel in ['train/weights/best.pt','train/results.csv','train/args.yaml','val_summary.json','test_summary.json','resolved_data.yaml','experiment_config.json']:
            p=folder/rel
            if p.is_file():z.write(p,arcname=name+'/'+rel)
print('Results:',result_zip)
files.download(str(result_zip))